# Spark at Scale: One Year of Taxi Trips

In the [pandas notebook](pandas_limits.ipynb) one year of taxi trips did not fit in memory. Here we analyze the **same year with Spark**, on **Databricks**.

**The data:** every taxi trip in New York City, published by the [NYC Taxi and Limousine Commission](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) as one Parquet file per month. In 2024 there were about **3.4 million trips per month**, **41 million** in the whole year. Each trip has 19 columns: pickup and drop-off time, distance, fare, tip, payment type, and more (see the [data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)).

Run the cells from top to bottom (**Shift + Enter**).

> ### Trivia: when "anonymous" taxi data was not anonymous
>
> In 2014 Chris Whong [obtained the 2013 taxi trips](https://chriswhong.com/open-data/foil_nyc_taxi/) from the city (about 173 million trips) with a freedom-of-information request. To protect the drivers, every taxi number and driver license number had been replaced by its **MD5 hash**.
>
> The engineer Vijay Pandurangan noticed that these numbers follow a short, known format, so there are only a few million possible values. Computing the hash of every possible value and comparing them with the data [took about two hours](https://tech.vijayp.ca/of-taxis-and-rainbows-f6bc289679a1) on one computer: every "anonymous" driver had a name again, with all their trips and income.
>
> Even without IDs, Anthony Tockar [matched paparazzi photos](https://agkn.wordpress.com/2014/09/15/riding-with-the-stars-passenger-privacy-in-the-nyc-taxicab-dataset/) of celebrities getting into taxis (time and place) with the trips, and found where they went and how much they tipped.
>
> **What we learn:** hashing is not anonymization when the possible values are few. And time + place can identify a person when combined with other information. The files we use today no longer contain taxi or driver numbers: only `VendorID`, the company that recorded the trip.

## 0. Setup: a place for the files

On Databricks, files are stored in a **volume** (a folder managed by Databricks). The next cell creates one called `taxi_data`, if it does not exist yet.

In [ ]:
import os
import time
import urllib.request

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.taxi_data")
DATA_DIR = "/Volumes/workspace/default/taxi_data"

### Measuring memory
As in the pandas notebook, we measure the memory used by **this notebook** (its Python process) and by the machine it runs on. The difference with pandas: here the data is processed by the Spark **workers**, and only the (small) results come back to the notebook.

In [ ]:
import psutil

GB = 1024 ** 3

def show_memory(label=""):
    """Memory of this notebook (its Python process) and of the whole machine."""
    notebook = psutil.Process().memory_info().rss
    m = psutil.virtual_memory()
    print(f"{label}this notebook: {notebook / GB:.1f} GB | machine: {m.used / GB:.1f} GB used, {m.available / GB:.1f} GB free of {m.total / GB:.1f} GB")

show_memory("start: ")

### Download the 12 months of 2024
About 650 MB in total: it takes a few minutes the first time. If you run the cell again, files already downloaded are skipped.

In [ ]:
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

start = time.perf_counter()
for month in range(1, 13):
    name = f"yellow_tripdata_2024-{month:02d}.parquet"
    path = f"{DATA_DIR}/{name}"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE_URL}/{name}", path)
    print(f"{name}: {os.path.getsize(path) / 1024 ** 2:.0f} MB")
print(f"done in {time.perf_counter() - start:.0f} s")

## 1. Read the whole year

One line reads all 12 files: the `*` means "every file whose name matches". The cell returns immediately, because Spark is **lazy**: it only looks at the column names, it does not read the data yet.

In [ ]:
trips = spark.read.parquet(f"{DATA_DIR}/yellow_tripdata_2024-*.parquet")
trips.printSchema()
show_memory("after reading the year: ")

In [ ]:
start = time.perf_counter()
print(f"trips in 2024: {trips.count():,} | time: {time.perf_counter() - start:.1f} s")

## 2. The same question as in pandas, on the whole year

Average tip by payment type, on **41 million** trips. In pandas we could not even load the data; here it takes a few seconds.

In [ ]:
start = time.perf_counter()
tips = (trips
    .groupBy("payment_type")
    .agg(F.count("*").alias("trips"), F.round(F.avg("tip_amount"), 2).alias("avg_tip"))
    .orderBy("payment_type"))
tips.show()
print(f"time: {time.perf_counter() - start:.1f} s")
show_memory("after the query: ")

Payment types: 0 = flex fare, 1 = credit card, 2 = cash, 3 = no charge, 4 = dispute, 5 = unknown, 6 = voided trip. Tips paid in cash are not recorded, so the average for cash is (almost) 0.

## 3. Why is Spark so fast? It reads only what it needs

Parquet is a **columnar** format: each column is stored separately in the file. Before running a query, Spark looks at it and reads **only the columns it needs**. Our tip question uses 2 columns out of 19.

Spark writes down its plan for every query. The plan is long and technical, so the next cell defines a small function that finds the two things we care about in it: **which columns** Spark reads, and **which filters** it applies while reading. (If you are curious, you can see the full plan with `tips.explain()`.)

In [ ]:
import contextlib
import io
import re

def what_spark_reads(df):
    """Print which columns and filters Spark uses when it reads the files for this query."""
    buffer = io.StringIO()
    with contextlib.redirect_stdout(buffer):
        df.explain()
    plan = buffer.getvalue()

    schema = re.search(r"ReadSchema: struct<(.*?)>", plan)
    if schema:
        columns = [c.split(":")[0] for c in schema.group(1).split(",")]
        print(f"columns read: {len(columns)} of {len(trips.columns)} -> {', '.join(columns)}")
    else:
        print("The plan has a different format here: run df.explain() and look for ReadSchema.")
    filters = re.search(r"PushedFilters: \[(.*?)\]", plan)
    if filters and filters.group(1):
        print(f"filters applied while reading the files: {filters.group(1)}")

what_spark_reads(tips)

Spark can also skip data with a **filter**. For example, only the trips longer than 100 miles: the filter is sent to the file reader, which can skip whole parts of the files that cannot contain such trips (`GreaterThan(trip_distance,100.0)` means *trip_distance > 100*).

In [ ]:
long_trips = trips.filter(F.col("trip_distance") > 100).select("tpep_pickup_datetime", "trip_distance", "total_amount")

what_spark_reads(long_trips)
print(f"trips longer than 100 miles: {long_trips.count():,}")

### How much data did Spark actually read?
A Parquet file knows how many bytes each column takes. The next cell adds them up for the 12 files, so we can compare the whole year with what our queries really need.

(Databricks also shows this under each cell: click **See performance** to find *Bytes read* and *Rows read*.)

In [ ]:
import pyarrow.parquet as pq

def megabytes_on_disk(columns=None):
    """Size on disk of the given columns (all columns if None) in the 12 files."""
    total = 0
    for month in range(1, 13):
        meta = pq.read_metadata(f"{DATA_DIR}/yellow_tripdata_2024-{month:02d}.parquet")
        for g in range(meta.num_row_groups):
            group = meta.row_group(g)
            for c in range(group.num_columns):
                column = group.column(c)
                if columns is None or column.path_in_schema in columns:
                    total += column.total_compressed_size
    return total / 1024 ** 2

print(f"the whole year, all 19 columns: {megabytes_on_disk():.0f} MB")
print(f"tip question (payment_type, tip_amount): {megabytes_on_disk(['payment_type', 'tip_amount']):.0f} MB")
print(f"long trips (3 columns): {megabytes_on_disk(['tpep_pickup_datetime', 'trip_distance', 'total_amount']):.0f} MB")

For the tip question Spark reads about **48 MB** from disk. pandas had to read all the files and unpack them into about **7 GB** of memory, and still could not merge them.

## 4. One billion rows

The taxi year is "only" 41 million rows. What about **one billion**? `spark.range` creates a DataFrame with the numbers from 0 to 999,999,999, and we compute some statistics on it.

In pandas, just one column of one billion numbers would need **8 GB** of memory, before doing anything with it. Spark never builds the whole table in one place: every machine generates and processes its own part.

In [ ]:
start = time.perf_counter()
billion = spark.range(1_000_000_000)

(billion
    .withColumn("is_even", (F.col("id") % 2 == 0))
    .groupBy("is_even")
    .agg(F.count("*").alias("rows"), F.sum("id").alias("sum"))
    .show())
print(f"time: {time.perf_counter() - start:.1f} s")
show_memory("after one billion rows: ")

## 5. Your turn

Answer these questions on the **whole year**. Some trips have wrong dates (for example in 2002 or 2009): keep only the trips of 2024 where it matters.

1. What is the **busiest hour** of the day? (hint: `F.hour("tpep_pickup_datetime")`)
2. Which **month** has the most trips? (hint: `F.month(...)`)
3. How many trips **paid by credit card** have a tip of more than 20% of the fare (`fare_amount`)?
4. **Airport trips**: how many trips per month paid an airport fee (`Airport_fee > 0`)?
5. **Data quality**: how many trips have a distance of 0, or a negative `total_amount`? What would you do with them?

In [ ]:
# Question 1

In [ ]:
# Question 2

In [ ]:
# Question 3

In [ ]:
# Question 4

In [ ]:
# Question 5

## 6. When NOT to use Spark

Spark is not always the best choice. Every Spark query has a **fixed cost**: Spark has to plan the query, split it into tasks, send them to the workers and collect the results. On big data this cost is tiny compared to the work. On **small** data it is most of the time, and pandas, which just runs the calculation in memory, is much faster.

To see it, we take a small table of **10,000 rows** that is already in memory, and run the same question with both. We run each one 3 times and keep the best time, so that the first (slower) run does not count.

In [ ]:
import pandas as pd

small_pd = pd.read_parquet(f"{DATA_DIR}/yellow_tripdata_2024-01.parquet", columns=["payment_type", "tip_amount"]).head(10_000)
small_spark = spark.createDataFrame(small_pd)

# Run a function a few times and return the best time, in milliseconds
def best_time(run, repeat=3):
    times = []
    for _ in range(repeat):
        start = time.perf_counter()
        run()
        times.append(time.perf_counter() - start)
    return min(times) * 1000

pandas_ms = best_time(lambda: small_pd.groupby("payment_type")["tip_amount"].mean())
spark_ms = best_time(lambda: small_spark.groupBy("payment_type").agg(F.avg("tip_amount")).collect())

print(f"10,000 rows | pandas: {pandas_ms:.1f} ms | Spark: {spark_ms:.0f} ms")

On 10,000 rows pandas answers in less than a millisecond, while Spark needs hundreds of times longer just to organize the work. With 41 million rows the situation is the opposite: pandas cannot even hold the data.

## pandas vs Spark on the same year

| | pandas (Colab) | Spark (Databricks) |
|---|---|---|
| memory of the notebook | grows month by month to about **7 GB** | stays about the same: the data never comes to the notebook |
| data read for the tip question | all the files (661 MB), unpacked to about 7 GB in memory | **48 MB**: only the 2 columns the question needs |
| the whole year as one table | not possible: not enough memory to merge | no problem, in a few seconds |
| one billion rows | 8 GB for a single column | a few seconds |

Of course the Spark workers also use memory. But each worker processes one piece of the data at a time (like the blocks of the cluster in Lesson 6), so the memory needed does not grow with the size of the data the way it does in pandas.

## Summary

- **pandas**: one machine, all the data in memory. Simple and fast, as long as the data fits.
- **Spark**: the data is split across many machines, read lazily, and only the columns and rows a query needs are read. The same code works on 1 million or 1 billion rows.
- **Rule of thumb**: if your data fits comfortably in memory, use pandas. If it does not, or if it will keep growing, use Spark.